# **LangChain `create_agent`**

---

## **1. 설치 및 환경 설정**

```bash
pip install langchain langchain-openai langchain-tavily langgraph
```



In [ ]:
from dotenv import load_dotenv
load_dotenv()

## **2. 기본 사용법**

In [ ]:
from langchain.agents import create_agent

# 기본 에이전트 생성 (도구 없음)
agent = create_agent(
    model="openai:gpt-4.1-nano",
    tools=[],
    system_prompt="You are a helpful assistant."
)

# 에이전트 실행
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "안녕하세요!"}
    ]
})

print(result["messages"][-1].content)

## **3. 도구(Tools) 사용**

`(1) 기본 Tavily 검색`

In [ ]:
from langchain.agents import create_agent
from langchain_tavily import TavilySearch

# Tavily 검색 도구 초기화
search_tool = TavilySearch(
    max_results=5,                    # 최대 검색 결과 수
    topic="general",                  # 검색 주제
)

# 검색 도구를 포함한 에이전트 생성
agent = create_agent(
    model="openai:gpt-4.1-nano",
    tools=[search_tool],
    system_prompt="You are a helpful research assistant that can search the web."
)

# 에이전트 실행
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "2024년 노벨 물리학상 수상자는 누구인가요?"}
    ]
})

print(result["messages"][-1].content)

`(2) 동적 파라미터 설정`

- 에이전트가 검색 시 동적으로 파라미터를 설정 가능

In [ ]:
from langchain.agents import create_agent
from langchain_tavily import TavilySearch

search_tool = TavilySearch(
    max_results=5,
)

agent = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[search_tool],
    system_prompt="""You are a research assistant. 
    When searching for academic content, use include_domains=['wikipedia.org'].
    For news, use topic='news'."""
)

In [ ]:
# 에이전트가 자동으로 적절한 검색 파라미터를 선택 (Wikipedia 검색)
result = agent.invoke({
    "messages": [
        {"role": "user", 
         "content": "Find information about quantum computing from Wikipedia only."}
    ]
})

for msg in result["messages"]:
    msg.pretty_print()

In [ ]:
# 에이전트가 동적으로 파라미터 설정 (최신 뉴스 검색)
result = agent.invoke({
    "messages": [
        {"role": "user", 
         "content": "Get me the latest news about artificial intelligence from the past week."}
    ]
})

for msg in result["messages"]:
    msg.pretty_print()

`(3) 순차적 도구 사용`

In [ ]:
from langchain_tavily import TavilySearch, TavilyExtract

# 검색 도구
search_tool = TavilySearch(
    max_results=5,
    topic="general"
)

# 콘텐츠 추출 도구
extract_tool = TavilyExtract(
    extract_depth="basic", 
    include_images=False
)

agent = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[search_tool, extract_tool],
    system_prompt="""You are a research assistant.
    - Use tavily_search to find relevant URLs
    - Use tavily_extract to get detailed content from specific URLs
    """
)

result = agent.invoke({
    "messages": [
        {"role": "user", 
         "content": "최신 AI 연구 동향을 찾아서 상세 내용을 추출해주세요."}
    ]
})

for msg in result["messages"]:
    msg.pretty_print()

## **4. Middleware 설정**

`(1) 대화 요약 Middleware`

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[search_tool],
    middleware=[
        SummarizationMiddleware(
            model="openai:gpt-4.1-nano",     # 요약에 사용할 모델
            max_tokens_before_summary=500,   # 요약 시작 임계값 (토큰 수)
            messages_to_keep=3,              # 요약 후에도 유지할 메시지 수
        )
    ],
    checkpointer=InMemorySaver(),
)

In [ ]:
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "최신 AI 연구 동향을 찾아서 설명해주세요."}
    ]
},
config={"configurable": {"thread_id": "custom_thread_001"}})

for msg in result["messages"]:
    msg.pretty_print()

In [ ]:
# 대화 계속
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "LLM과 생성형 AI를 중심으로 자세하게 조사해주세요."}
    ]
},
config={"configurable": {"thread_id": "custom_thread_001"}})

for msg in result["messages"]:
    msg.pretty_print()

In [ ]:
# 대화 계속
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "에너지 부족 문제에 대해 조사해주세요."}
    ]
},
config={"configurable": {"thread_id": "custom_thread_001"}})

for msg in result["messages"]:
    msg.pretty_print()

`(2) 커스텀 Middleware 생성`

In [ ]:
from langchain.agents.middleware import AgentMiddleware
from typing import Any, Dict

class LoggingMiddleware(AgentMiddleware):
    """모든 도구 호출을 로깅하는 Middleware"""
    
    def before_model(self, state: Dict[str, Any], runtime) -> Dict[str, Any] | None:
        """모델 호출 전에 실행"""
        print(f"🤖 모델 호출 전: {len(state['messages'])} 메시지")
        return None
    
    def after_model(self, state: Dict[str, Any], runtime) -> Dict[str, Any] | None:
        """모델 호출 후에 실행"""
        last_message = state["messages"][-1]
        if hasattr(last_message, "tool_calls") and last_message.tool_calls:
            print(f"🔧 도구 호출: {[tc['name'] for tc in last_message.tool_calls]}")
        return None

class RateLimitMiddleware(AgentMiddleware):
    """도구 호출 속도 제한"""
    
    def __init__(self, max_calls_per_minute: int = 10):
        self.max_calls = max_calls_per_minute
        self.call_times = []
    
    def wrap_tool_call(self, request, handler):
        """도구 호출을 래핑하여 속도 제한 적용"""
        import time
        from datetime import datetime, timedelta
        
        # 최근 1분 내 호출 정리
        now = datetime.now()
        self.call_times = [t for t in self.call_times 
                          if now - t < timedelta(minutes=1)]
        
        # 속도 제한 확인
        if len(self.call_times) >= self.max_calls:
            raise Exception(f"속도 제한: 분당 최대 {self.max_calls}회 호출 가능")
        
        # 도구 실행
        self.call_times.append(now)
        return handler(request)

agent = create_agent(
    model="openai:gpt-5-nano",
    tools=[search_tool],
    middleware=[
        LoggingMiddleware(),
        RateLimitMiddleware(max_calls_per_minute=1) # 테스트용으로 1분에 1회 제한
    ]
)

In [ ]:
# 에이전트 실행 (로깅 확인)
result = agent.invoke({
    "messages": [{"role": "user", "content": "2024년 노벨 물리학상 수상자는 누구인가요?"}]
})

for msg in result["messages"]:
    msg.pretty_print()

In [ ]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "2020년대 노벨 물리학상 수상자는 누구인가요?"}]
})

for msg in result["messages"]:
    msg.pretty_print()

`(3) 대화 영속성 (Checkpointing)`

- Checkpointing을 통해 대화의 상태를 저장하고 복원 가능

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent

# 메모리 체크포인터 생성
checkpointer = InMemorySaver()

agent = create_agent(
    model="openai:gpt-4.1-nano",
    tools=[search_tool],
    checkpointer=checkpointer
)

# 첫 번째 대화
config1 = {"configurable": {"thread_id": "user-123-session-1"}}
result1 = agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름은 김철수야"}]},
    config1
)
for msg in result1["messages"]:
    msg.pretty_print()

In [ ]:
# 두 번째 대화 (같은 thread_id로 이전 대화 기억)
result2 = agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름이 뭐였지?"}]},
    config1
)
for msg in result2["messages"]:
    msg.pretty_print()  # "김철수"라고 기억함

In [ ]:
# 다른 대화 (다른 thread_id로 새로운 세션)
config2 = {"configurable": {"thread_id": "user-123-session-2"}}
result3 = agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름이 뭐야?"}]},
    config2
)
for msg in result3["messages"]:
    msg.pretty_print()  # 이전 대화 기억 없음

## **5. 실전 프로젝트**

`(1) 웹 리서치 에이전트`

In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch, TavilyExtract
from langgraph.checkpoint.memory import InMemorySaver

# 도구 설정
search_tool = TavilySearch(
    max_results=5,
    topic="general",
    search_depth="advanced"
)

extract_tool = TavilyExtract(
    extract_depth="advanced"
)

# 모델 인스턴스를 직접 생성하여 reasoning 설정
model = ChatOpenAI(
    model="gpt-5-nano",  # reasoning 지원 모델
    reasoning_effort="low",  # 'low', 'medium', 'high'
    temperature=0.3,  
    max_completion_tokens=5000
)

# 에이전트 생성
research_agent = create_agent(
    model=model,  # 모델 인스턴스 전달
    tools=[search_tool, extract_tool],   # 도구 등록
    system_prompt="""You are an expert research assistant.

When conducting research:
1. Use tavily_search to find relevant sources
2. Evaluate the credibility of sources
3. Use tavily_extract to get detailed information from the most relevant URLs
4. Synthesize information from multiple sources
5. Provide citations for all claims

Format your response as:
## Research Summary
[Brief overview]

## Key Findings
- Finding 1 [Source]
- Finding 2 [Source]

## Detailed Analysis
[In-depth analysis]

## Sources
- [URL 1]
- [URL 2]
""",
    checkpointer=InMemorySaver()
)

# 실행
result = research_agent.invoke({
    "messages": [
        {"role": "user", 
         "content": "2024년 AI 산업의 주요 트렌드를 조사하고 분석해주세요."}
    ]
}, {"configurable": {"thread_id": "research-1"}})

print(result["messages"][-1].content)

`(2) SQL 데이터베이스 에이전트`

In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain_community.utilities import SQLDatabase

# 데이터베이스 연결
db = SQLDatabase.from_uri("sqlite:///sqlite-sakila.db")

@tool
def list_tables() -> str:
    """데이터베이스의 모든 테이블을 나열합니다."""
    return db.get_table_names()

@tool
def get_schema(table_name: str) -> str:
    """특정 테이블의 스키마를 가져옵니다."""
    return db.get_table_info([table_name])

@tool
def run_query(query: str) -> str:
    """SQL 쿼리를 실행합니다."""
    try:
        result = db.run(query)
        return result
    except Exception as e:
        return f"Error: {str(e)}"


model = ChatOpenAI(
    model="gpt-5-mini",
    reasoning_effort="low",
    temperature=0,
    max_completion_tokens=2000
)   

sql_agent = create_agent(
    model=model,
    tools=[list_tables, get_schema, run_query],
    system_prompt="""You are a SQL expert assistant.

When answering questions about the database:
1. First, list available tables using list_tables
2. Check relevant table schemas using get_schema
3. Generate and execute SQL queries using run_query
4. Double-check queries for correctness
5. Present results in a clear format

Safety guidelines:
- Only use SELECT queries
- Never modify data (no INSERT, UPDATE, DELETE)
- Validate table and column names before querying
"""
)

# 실행
result = sql_agent.invoke({
    "messages": [
        {"role": "user", "content": "가장 많이 대여한 고객은 누구인가요?"}
    ]
})

for msg in result["messages"]:
    msg.pretty_print()

In [ ]:
# 실행
result = sql_agent.invoke({
    "messages": [
        {"role": "user", "content": "가장 인기가 많은 영화 5편을 알려주세요."}
    ]
})

for msg in result["messages"]:
    msg.pretty_print()